# 🪷 CHÙA ONLINE - TIKTOK LIVE INTERACTIVE GAME

Notebook này giúp bạn khởi chạy toàn bộ hệ thống **TikTok Live Connector & Màn hình 3D Chùa Online** trực tiếp trên Google Colab miễn phí.

Sau khi chạy, hệ thống sẽ cấp cho bạn một **đường dẫn HTTPS công khai** (Public URL). Bạn có thể:
1. Mở xem trực tiếp trên trình duyệt hoặc điện thoại.
2. Dán link vào **OBS Studio -> Browser Source** trên máy tính cá nhân để phát sóng lên TikTok Live.


In [ ]:
# @title 📥 Bước 1: Tải mã nguồn & Cài đặt môi trường
# @markdown Chạy ô này để tải code và cài đặt các thư viện cần thiết.
import os
import subprocess

print("⏳ Đang tải mã nguồn và cài đặt dependencies...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)

%cd /content/app/connector
!npm install --silent

# Cài đặt cloudflared để tạo đường link HTTPS công khai miễn phí không cần đăng ký tài khoản
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("
✅ CÀI ĐẶT HOÀN TẤT! Hãy chuyển sang Bước 2 bên dưới.")


In [ ]:
# @title 🚀 Bước 2: Cấu hình & Khởi chạy Server Chùa Online
# @markdown Hãy chọn chế độ chạy và bấm nút Run:

Che_Do_Chay = "Mock Mode (Test giả lập sự kiện)" # @param ["Mock Mode (Test giả lập sự kiện)", "TikTok Live Thật"]
TikTok_Username = "zuno_live" # @param {type:"string"}

import time
import re
from IPython.display import display, HTML

# Tắt các tiến trình cũ nếu có
!pkill -f node > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

is_mock = "true" if "Mock" in Che_Do_Chay else "false"
env_vars = os.environ.copy()
env_vars["MOCK_MODE"] = is_mock
env_vars["TIKTOK_USERNAME"] = TikTok_Username
env_vars["HTTP_PORT"] = "3000"
env_vars["WS_PORT"] = "8080"

# Khởi chạy Node.js server chạy nền
print(f"⏳ Đang khởi động Server (Chế độ: {'GIẢ LẬP MOCK' if is_mock == 'true' else f'KÊNH @{TikTok_Username}'})...")
server_process = subprocess.Popen(["node", "src/index.js"], cwd="/content/app/connector", env=env_vars, stdout=open("/tmp/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)

# Mở Cloudflare Tunnel để tạo link công khai
print("🌐 Đang tạo đường dẫn HTTPS công khai...")
tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
start_time = time.time()
while time.time() - start_time < 20:
    line = tunnel_process.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.1)

if public_url:
    client_url = f"{public_url}/client"
    dashboard_url = public_url
    html_banner = f"""
    <div style="background: #1a0f0a; border: 2px solid #f1c40f; border-radius: 12px; padding: 20px; color: #fff; font-family: sans-serif;">
        <h2 style="color: #f1c40f; margin-top: 0;">🎉 HỆ THỐNG ĐÃ SẴN SÀNG HOẠT ĐỘNG!</h2>
        <p>Bấm vào các liên kết bên dưới để mở giao diện:</p>
        <div style="margin: 15px 0;">
            <a href="{client_url}" target="_blank" style="display: inline-block; background: #e67e22; color: #fff; padding: 12px 20px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 15px; margin-right: 10px;">🪷 MỞ MÀN HÌNH CHÙA 3D (Tỉ lệ 9:16)</a>
            <a href="{dashboard_url}" target="_blank" style="display: inline-block; background: #27ae60; color: #fff; padding: 12px 20px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 15px;">🎮 BẢNG ĐIỀU KHIỂN & TEST NÚT BẤM</a>
        </div>
        <p style="color: #ccc; font-size: 13px;">
            💡 <strong>Dành cho OBS Studio:</strong> Thêm <i>Browser Source</i> trong OBS, dán link <code>{client_url}</code>, đặt <b>Width: 1080</b>, <b>Height: 1920</b>.
        </p>
    </div>
    """
    display(HTML(html_banner))
else:
    print("⚠️ Chưa lấy được URL Cloudflare. Đang kiểm tra log:")
    !cat /tmp/server.log


In [ ]:
# @title 📜 Bước 3: Xem Log hoạt động thời gian thực (Xem ai donate, chat...)
!tail -n 25 /tmp/server.log


In [ ]:
# @title ⏹️ Bước 4: Dừng hệ thống khi không sử dụng nữa
!pkill -f node
!pkill -f cloudflared
print("🛑 Đã dừng toàn bộ dịch vụ Chùa Online an toàn!")
